In [ ]:
import numpy as np
import os
import scipy
import mat73
from load_data_function import load_data,save_data
import re
from load_data_function import fig_plot,battery_soh_plot,smooth_soh
import matplotlib.pyplot as plt

In [ ]:
test_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\original_battery_data\Toyota_MIT_dataset/2017-05-12_batchdata_updated_struct_errorcorrect.mat'
#data= mat73.loadmat(test_path)


In [ ]:
#print(data.keys())

In [ ]:
#print(data['batch'].keys())

In [ ]:
#print(data['batch']['cycles'][0].keys())

In [ ]:
Toyota_MIT_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model\original_battery_data\Toyota_MIT_dataset'
"""def get_immediate_subdirs(parent_dir):
    # 获取所有条目名称，并过滤出子文件夹
    return [name for name in os.listdir(parent_dir)
            if os.path.isdir(os.path.join(parent_dir, name))]
"""
package_list=os.listdir(Toyota_MIT_path)
print(package_list)


Toyota_MIT_data={}
Toyota_MIT_SOH={}
for i,package in enumerate(package_list):
    package_data=mat73.loadmat(Toyota_MIT_path+'/'+package)
    print(f'package: {package}')
    battery_list=package_data['batch']['policy']
    print(battery_list)

    package1={}
    package2={}
    for j,battery in enumerate(battery_list):
        print(f'battery: {battery}')
        battery_data=package_data['batch']['cycles'][j]
        if j==0:
            #print(len(battery_data['I']))

        voltage=[]
        current=[]
        time=[]
        capacity=[]
        package1[f'battery_{j+1}']=[]
        package2[f'battery_{j+1}']=[]
        for k in range(len(battery_data['I'])-1):
            voltage=battery_data['V'][k+1].reshape(1,-1)
            #print(voltage.shape)
            current=battery_data['I'][k+1].reshape(1,-1)
            time_segment = battery_data['t'][k+1].reshape(1,-1)
            time=60 * time_segment  # 转换为秒
            capacity=package_data['batch']['summary'][j]['QDischarge'][k+1]

            soh=capacity/1.1
            package1[f'battery_{j+1}'].append(np.concatenate((voltage,current,time),axis=0))
            package2[f'battery_{j+1}'].append(soh)
    Toyota_MIT_data[f'package_{i+1}']=package1
    Toyota_MIT_SOH[f'package_{i+1}']=package2

In [ ]:
print(Toyota_MIT_data.keys())

In [ ]:
print(Toyota_MIT_data['package_1'].keys())

In [ ]:
print(Toyota_MIT_data['package_1']['battery_1'][0].shape)

In [ ]:
print(Toyota_MIT_SOH.keys())

In [ ]:
print(Toyota_MIT_SOH['package_1'].keys())

In [ ]:
print(Toyota_MIT_SOH['package_1']['battery_1'][0])

In [ ]:
fig_plot(Toyota_MIT_data['package_1']['battery_1'][0][0])
print(Toyota_MIT_data['package_1']['battery_1'][3][2][0:4])

In [ ]:
fig_plot(Toyota_MIT_SOH['package_3']['battery_16'])

In [ ]:
storage_path='D:\pycharm\Py_Projects/battery_SOH_predict/foundation_model/transformed_data\Toyota_MIT_dataset'
#save_data(Toyota_MIT_data, os.path.join(storage_path, 'Toyota_MIT_data.pkl'))
Toyota_MIT_data = load_data(os.path.join(storage_path, 'Toyota_MIT_data.pkl'))
#save_data(Toyota_MIT_SOH, os.path.join(storage_path, 'Toyota_MIT_SOH.pkl'))
Toyota_MIT_SOH = load_data(os.path.join(storage_path, 'Toyota_MIT_SOH.pkl'))

In [ ]:
plt.figure(figsize=(10,5))
package='package_1'
for key in Toyota_MIT_SOH[package].keys():
    plt.plot(Toyota_MIT_SOH[package][key])
plt.legend(list(Toyota_MIT_SOH[package].keys()))
plt.show()

In [ ]:
smooth_Toyota_MIT_SOH=smooth_soh(Toyota_MIT_SOH,'gaussian',sigma=1)

In [ ]:
plt.figure(figsize=(10,5))
package='package_1'
battery_soh_plot(smooth_Toyota_MIT_SOH,Toyota_MIT_SOH[package].keys(),package)

In [ ]:
save_data(Toyota_MIT_SOH, os.path.join(storage_path, 'Toyota_MIT_SOH.pkl'))